# 01 — Data Understanding & Profiling

## Purpose
This notebook performs **raw data understanding** for the Online Retail II dataset.

The goal is to:
- inspect the original Excel workbook
- understand workbook structure
- profile the raw data
- identify quality issues
- document what must be resolved before cleaning, database design, and dashboarding

## Important boundary
This notebook is for **observation and profiling only**.

It should answer:
- what the raw data contains
- what is wrong in the raw data
- what business meaning the columns have
- what issues must be addressed in cleaning

Cleaning and dataset creation belong in `02_data_cleaning.ipynb`.

In [8]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
file_path = project_root / "data" / "raw" / "online_retail_ii" / "online_retail_II.xlsx"

print("Current working directory:", Path.cwd())
print("Project root:", project_root)
print("File path:", file_path)
print("File exists:", file_path.exists())

Current working directory: /Users/pratikchetry/Desktop/retail-revenue-intelligence/notebooks
Project root: /Users/pratikchetry/Desktop/retail-revenue-intelligence
File path: /Users/pratikchetry/Desktop/retail-revenue-intelligence/data/raw/online_retail_ii/online_retail_II.xlsx
File exists: True


In [34]:
import hashlib

with open(file_path, "rb") as f:
    fingerprint = hashlib.md5(f.read()).hexdigest()

print("Source file MD5:", fingerprint)

Source file MD5: ed54ccfc5d358481c399cc11d0a244be


## Raw source file

The project uses the Online Retail II Excel workbook as the raw transaction source.

### Workbook structure
The file contains two yearly sheets:
- `Year 2009-2010`
- `Year 2010-2011`

Both sheets must be checked for:
- identical column structure
- compatible data types
- suitability for combination into one raw transaction dataset

In [9]:
excel_file = pd.ExcelFile(file_path)
for s in excel_file.sheet_names:
    print(repr(s))

'Year 2009-2010'
'Year 2010-2011'


In [10]:
excel_file = pd.ExcelFile(file_path)
excel_file.sheet_names

['Year 2009-2010', 'Year 2010-2011']

In [11]:
sheet_1 = "Year 2009-2010"
sheet_2 = "Year 2010-2011"

df_2009_2010 = pd.read_excel(file_path, sheet_name=sheet_1)
df_2010_2011 = pd.read_excel(file_path, sheet_name=sheet_2)

print("2009-2010 shape:", df_2009_2010.shape)
print("2010-2011 shape:", df_2010_2011.shape)

2009-2010 shape: (525461, 8)
2010-2011 shape: (541910, 8)


In [12]:
df_2009_2010.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [13]:
df_2010_2011.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [14]:
print("Columns in 2009-2010:")
print(df_2009_2010.columns.tolist())

print("\nColumns in 2010-2011:")
print(df_2010_2011.columns.tolist())

Columns in 2009-2010:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Columns in 2010-2011:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [15]:
same_columns = df_2009_2010.columns.tolist() == df_2010_2011.columns.tolist()
print("Do both sheets have the same columns?", same_columns)

Do both sheets have the same columns? True


In [16]:
df = pd.concat([df_2009_2010, df_2010_2011], ignore_index=True)

print("Combined shape:", df.shape)
df.head()

Combined shape: (1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## Combined raw dataset structure

After loading both sheets, the workbook produces one combined transaction dataset with:

- **1,067,371 rows**
- **8 columns**

### Raw columns
- `Invoice`
- `StockCode`
- `Description`
- `Quantity`
- `InvoiceDate`
- `Price`
- `Customer ID`
- `Country`

### Business meaning
- `Invoice` identifies transaction, cancellation, or adjustment activity
- `StockCode` identifies the item or operational code
- `Description` gives the product or entry description
- `Quantity` captures units sold or returned
- `InvoiceDate` provides the transaction timestamp
- `Price` is the unit price
- `Customer ID` enables customer-level analysis
- `Country` supports geographic analysis|

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


In [18]:
missing_values = df.isna().sum().sort_values(ascending=False)
print("Missing Values:", missing_values)

Missing Values: Customer ID    243007
Description      4382
Invoice             0
StockCode           0
Quantity            0
InvoiceDate         0
Price               0
Country             0
dtype: int64


In [19]:
duplicate_rows = df.duplicated().sum()
print("Duplicate rows:", duplicate_rows)

Duplicate rows: 34335


In [20]:
negative_quantity_rows = (df["Quantity"] < 0).sum()
negative_price_rows = (df["Price"] < 0).sum()

print("Negative Quantity rows:", negative_quantity_rows)
print("Negative Price rows:", negative_price_rows)

Negative Quantity rows: 22950
Negative Price rows: 5


In [21]:
df.describe(include="all")

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
count,1067371.0,1067371,1062989,1.067371e+06,1067371,1.067371e+06,824364.000000,1067371
unique,53628.0,5305,5698,NaN,NaN,NaN,NaN,43
top,537434.0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,NaN,NaN,NaN,NaN,United Kingdom
freq,1350.0,5829,5918,NaN,NaN,NaN,NaN,981330
mean,NaN,NaN,NaN,9.938898e+00,2011-01-02 21:13:55.394028544,4.649388e+00,15324.638504,NaN
min,NaN,NaN,NaN,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04,12346.000000,NaN
25%,NaN,NaN,NaN,1.000000e+00,2010-07-09 09:46:00,1.250000e+00,13975.000000,NaN
50%,NaN,NaN,NaN,3.000000e+00,2010-12-07 15:28:00,2.100000e+00,15255.000000,NaN
75%,NaN,NaN,NaN,1.000000e+01,2011-07-22 10:23:00,4.150000e+00,16797.000000,NaN
max,NaN,NaN,NaN,8.099500e+04,2011-12-09 12:50:00,3.897000e+04,18287.000000,NaN


In [22]:
print("Unique countries:", df["Country"].nunique())
print("Unique stock codes:", df["StockCode"].nunique())
print("Unique invoices:", df["Invoice"].nunique())
print("Unique customers:", df["Customer ID"].nunique())

Unique countries: 43
Unique stock codes: 5305
Unique invoices: 53628
Unique customers: 5942


In [23]:
df[df["Quantity"] < 0].head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,16321.0,Australia
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,16321.0,Australia
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,16321.0,Australia
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom


In [24]:
df[df["Price"] < 0].head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


## Profiling summary

The raw combined dataset contains the following quality issues:

- Missing `Customer ID`: **243,007**
- Missing `Description`: **4,382**
- Duplicate rows: **34,335**
- Negative `Quantity` rows: **22,950**
- Negative `Price` rows: **5**
- Unique customers: **5,942**
- Unique products (`StockCode`): **5,305**
- Unique countries: **43**

These issues directly affect:
- customer segmentation
- revenue interpretation
- product analysis
- database schema design
- dashboard logic

## Business interpretation of major raw data issues

### 1. Missing Customer ID
`Customer ID` is missing in **243,007** rows.

**Implication:**  
These rows may still contribute to total sales reporting, but they cannot directly support customer-level analytics such as RFM unless an explicit unknown-customer policy is defined later.

### 2. Missing Description
`Description` is missing in **4,382** rows.

**Implication:**  
Product-level reporting and dimension design will require review of these rows before deciding whether they should be retained, excluded, or categorized separately.

### 3. Negative Quantity
`Quantity` is negative in **22,950** rows.

**Implication:**  
These rows are likely returns or cancellations and should not be treated as normal positive sales.

### 4. Negative Price
`Price` is negative in only **5** rows.

**Implication:**  
These rows likely represent accounting adjustments rather than standard sales transactions.

### 5. Duplicate Rows
There are **34,335** exact duplicate rows.

**Implication:**  
Revenue and quantity totals will be overstated unless duplicates are removed during cleaning.

In [25]:
column_meaning = {
    "Invoice": "Transaction or invoice reference",
    "StockCode": "Product/item code",
    "Description": "Product description",
    "Quantity": "Number of units sold or returned",
    "InvoiceDate": "Transaction timestamp",
    "Price": "Unit price of the item",
    "Customer ID": "Unique customer identifier",
    "Country": "Customer or transaction country"
}

for col, meaning in column_meaning.items():
    print(f"{col}: {meaning}")

Invoice: Transaction or invoice reference
StockCode: Product/item code
Description: Product description
Quantity: Number of units sold or returned
InvoiceDate: Transaction timestamp
Price: Unit price of the item
Customer ID: Unique customer identifier
Country: Customer or transaction country


In [26]:
print("Sample invoices with negative quantity:")
display(df[df["Quantity"] < 0][["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID", "Country"]].head(10))

Sample invoices with negative quantity:


,Invoice,StockCode,Description,Quantity,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2.95,16321.0,Australia
183,C489449,21871,SAVE THE PLANET MUG,-12,1.25,16321.0,Australia
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,1.25,16321.0,Australia
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,0.85,16321.0,Australia
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2.95,16321.0,Australia
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,4.25,17592.0,United Kingdom


In [27]:
print("Sample invoices with negative price:")
display(df[df["Price"] < 0][["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID", "Country"]].head(10))

Sample invoices with negative price:


,Invoice,StockCode,Description,Quantity,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,-38925.87,NaN,United Kingdom
825444,A563186,B,Adjust bad debt,1,-11062.06,NaN,United Kingdom
825445,A563187,B,Adjust bad debt,1,-11062.06,NaN,United Kingdom


In [28]:
returns_like_rows = df[df["Invoice"].astype(str).str.startswith("C", na=False)]
print("Rows with invoice starting with 'C':", returns_like_rows.shape[0])
returns_like_rows.head(10)

Rows with invoice starting with 'C': 19494


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,16321.0,Australia
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,16321.0,Australia
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,16321.0,Australia
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,17592.0,United Kingdom


In [29]:
negative_quantity_with_c_invoice = df[
    (df["Quantity"] < 0) & (df["Invoice"].astype(str).str.startswith("C", na=False))
].shape[0]

negative_quantity_total = (df["Quantity"] < 0).sum()

print("Negative quantity rows total:", negative_quantity_total)
print("Negative quantity rows with invoice starting 'C':", negative_quantity_with_c_invoice)
print("Percentage:", round((negative_quantity_with_c_invoice / negative_quantity_total) * 100, 2), "%")

Negative quantity rows total: 22950
Negative quantity rows with invoice starting 'C': 19493
Percentage: 84.94 %


In [30]:
negative_price_rows = df[df["Price"] < 0]
negative_price_rows[["Invoice", "StockCode", "Description", "Quantity", "Price", "Customer ID", "Country"]]

,Invoice,StockCode,Description,Quantity,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,-38925.87,NaN,United Kingdom
825444,A563186,B,Adjust bad debt,1,-11062.06,NaN,United Kingdom
825445,A563187,B,Adjust bad debt,1,-11062.06,NaN,United Kingdom


In [31]:
missing_description_rows = df[df["Description"].isna()]
print("Missing Description rows:", missing_description_rows.shape[0])
missing_description_rows.head(10)

Missing Description rows: 4382


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.0,NaN,United Kingdom
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.0,NaN,United Kingdom
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.0,NaN,United Kingdom
4566,489821,85049G,NaN,-240,2009-12-02 13:25:00,0.0,NaN,United Kingdom
6378,489882,35751C,NaN,12,2009-12-02 16:22:00,0.0,NaN,United Kingdom
6555,489898,79323G,NaN,954,2009-12-03 09:40:00,0.0,NaN,United Kingdom
6576,489901,21098,NaN,-200,2009-12-03 09:47:00,0.0,NaN,United Kingdom
6581,489903,21166,NaN,48,2009-12-03 09:57:00,0.0,NaN,United Kingdom


In [32]:
missing_customer_rows = df[df["Customer ID"].isna()]
print("Missing Customer ID rows:", missing_customer_rows.shape[0])
missing_customer_rows.head(10)

Missing Customer ID rows: 243007


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom
578,489525,85227,SET/6 3D KIT CARDS FOR KIDS,1,2009-12-01 11:49:00,0.85,NaN,United Kingdom
1055,489548,22271,FELTCRAFT DOLL ROSIE,1,2009-12-01 12:32:00,2.95,NaN,United Kingdom
1056,489548,22254,FELT TOADSTOOL LARGE,12,2009-12-01 12:32:00,1.25,NaN,United Kingdom
1057,489548,22273,FELTCRAFT DOLL MOLLY,3,2009-12-01 12:32:00,2.95,NaN,United Kingdom
1058,489548,22195,LARGE HEART MEASURING SPOONS,1,2009-12-01 12:32:00,1.65,NaN,United Kingdom


## Returns and adjustment profiling

A large share of negative quantity rows appear to be linked to invoice numbers beginning with `C`, which strongly suggests return or cancellation behavior rather than random bad data.

Negative price rows appear to be accounting-style entries such as bad debt adjustments rather than normal sales transactions.

### Why this matters
This means the cleaning stage must explicitly separate:
- positive sales
- returns/cancellations
- accounting adjustments

These cannot be mixed into one reporting table without causing misleading revenue logic.

## Design implications before cleaning

This profiling stage confirms that the following decisions must be made explicitly in the cleaning notebook:

1. how to remove duplicates
2. how to separate returns/cancellations from positive sales
3. how to isolate accounting adjustments
4. how to treat missing `Customer ID`
5. how to treat missing `Description`
6. how to identify non-merchandise stock codes
7. what will become the final staging datasets

These decisions will later affect:
- PostgreSQL schema design
- SQL views
- Tableau dashboard logic
- Power BI / DAX interpretation
- validation benchmarks

In [33]:
if "df" not in globals():
    raise ValueError("Dataframe 'df' is not defined yet. Run the data loading and combine cells first.")

profile_summary = {
    "total_rows": int(len(df)),
    "total_columns": int(len(df.columns)),
    "duplicate_rows": int(df.duplicated().sum()),
    "missing_customer_id": int(df["Customer ID"].isna().sum()),
    "missing_description": int(df["Description"].isna().sum()),
    "negative_quantity_rows": int((df["Quantity"] < 0).sum()),
    "negative_price_rows": int((df["Price"] < 0).sum()),
    "date_range_min": str(df["InvoiceDate"].min()),
    "date_range_max": str(df["InvoiceDate"].max()),
    "unique_customers": int(df["Customer ID"].nunique()),
    "unique_products": int(df["StockCode"].nunique()),
    "unique_countries": int(df["Country"].nunique())
}

profile_summary

{'total_rows': 1067371,
 'total_columns': 8,
 'duplicate_rows': 34335,
 'missing_customer_id': 243007,
 'missing_description': 4382,
 'negative_quantity_rows': 22950,
 'negative_price_rows': 5,
 'date_range_min': '2009-12-01 07:45:00',
 'date_range_max': '2011-12-09 12:50:00',
 'unique_customers': 5942,
 'unique_products': 5305,
 'unique_countries': 43}

In [35]:
suspicious_keywords = [
    "postage", "manual", "dotcom", "bank charges",
    "amazon", "fee", "adjust", "sample", "test"
]

desc_lower = df["Description"].str.lower().fillna("")
mask = desc_lower.str.contains("|".join(suspicious_keywords), na=False)

non_merchandise_scan = (
    df[mask]
    .groupby(["StockCode", "Description"])
    .size()
    .reset_index(name="row_count")
    .sort_values("row_count", ascending=False)
)

print("Potential non-merchandise rows:")
display(non_merchandise_scan.head(20))

Potential non-merchandise rows:


,StockCode,Description,row_count
167,POST,POSTAGE,2115
165,DOT,DOTCOM POSTAGE,1444
166,M,Manual,1421
45,22301,COFFEE MUG CAT + BIRD DESIGN,706
102,37370,RETRO COFFEE MUGS ASSORTED,579
44,22300,COFFEE MUG DOG + BALL DESIGN,573
47,22303,COFFEE MUG APPLES DESIGN,571
10,21216,"SET 3 RETROSPOT TEA,COFFEE,SUGAR",535
87,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,509
46,22302,COFFEE MUG PEARS DESIGN,411


## Result — initial non-merchandise scan

The profiling scan surfaced several stock codes and descriptions that may not represent standard merchandise.

### Confirmed high-priority non-merchandise candidates
The strongest candidates identified at this stage are:

- `POST` → **POSTAGE** (`2115` rows)
- `DOT` → **DOTCOM POSTAGE** (`1444` rows)
- `M` → **Manual** (`1421` rows)

### Important caution
The keyword scan also returned false positives, especially where words like **coffee** appear inside normal product descriptions such as mugs, tins, candles, and containers.

### Business implication
This means non-merchandise detection cannot rely only on keyword matching.  
A final reference list must be created during cleaning so that operational/service entries are separated from real merchandise before dashboard product analysis begins.

### Design implication
The cleaning notebook must produce an explicit `non_merchandise_codes.csv` reference file, and the PostgreSQL product dimension must later support fields such as:
- `is_merchandise`
- `product_type`

In [36]:
import json

output_path = project_root / "outputs" / "reports" / "profiling_summary.json"
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w") as f:
    json.dump(profile_summary, f, indent=2)

print("Profiling summary saved to:", output_path)

Profiling summary saved to: /Users/pratikchetry/Desktop/retail-revenue-intelligence/outputs/reports/profiling_summary.json


## Result — profiling artifacts saved

The raw profiling stage has now produced two permanent reference outputs:

- **Source file fingerprint (MD5):** `ed54ccfc5d358481c399cc11d0a244be`
- **Profiling summary file:** `outputs/reports/profiling_summary.json`

### Business implication
These artifacts make the profiling stage reproducible and auditable. The project can now prove:
- which exact raw file was used
- what the raw baseline counts and issues were before cleaning began

### Design implication
All later cleaning, PostgreSQL loading, and validation steps should reconcile against this saved profiling baseline rather than relying only on notebook memory.

## Conclusion of the profiling stage

The raw Online Retail II dataset is suitable for a strong retail analytics project, but it contains structural issues that must be resolved before database loading and dashboarding.

### Profiling established the following baseline facts
- the raw workbook contains **1,067,371 rows**
- exact duplicates exist and must be removed
- returns/cancellations and accounting adjustments must be separated from normal sales
- missing customer IDs affect customer-level analysis
- non-merchandise stock codes exist and must be flagged explicitly
- the project now has a saved profiling baseline and raw file fingerprint for future validation

The next step is:
**controlled cleaning with an audit trail in `02_data_cleaning.ipynb`.**